In [1]:
import numpy as np
import pandas as pd
import mne
from pathlib import Path

# ============================================================
# 1. EXPERIMENTAL DESIGN
# ============================================================

conditions = [
    ("_BL", "_plan", ""),
    ("_BL", "_go", ""),
    ("_MAIN", "_plan", "_baseline"),
    ("_MAIN", "_go", "_baseline"),
    ("_MAIN", "_plan", "_adaptation"),
    ("_MAIN", "_go", "_adaptation"),
]

# PILOT: one participant and one condition
sub_name = "s1_pac_sub01"
task = "_BL"
task_stage = "_go"
block = ""

n_surrogates = 20

# ============================================================
# 2. ROI DEFINITIONS
# ============================================================

roi_label_names = [
    # Primary Motor Cortex
    "G_precentral-lh", "G_precentral-rh",
    "S_central-lh", "S_central-rh",

    # Premotor Cortex
    "S_precentral-sup-part-lh",
    "S_precentral-sup-part-rh",
    "S_precentral-inf-part-lh",
    "S_precentral-inf-part-rh",
    "G_front_middle-lh",
    "G_front_middle-rh",

    # Supplementary Motor Area
    "G_and_S_paracentral-lh",
    "G_and_S_paracentral-rh",
    "G_front_sup-lh",
    "G_front_sup-rh",

    # Primary Somatosensory Cortex
    "G_postcentral-lh",
    "G_postcentral-rh",
    "S_postcentral-lh",
    "S_postcentral-rh",

    # Prefrontal Cortex
    "S_front_sup-lh",
    "S_front_sup-rh",
    "S_front_inf-lh",
    "S_front_inf-rh",
    "S_front_middle-lh",
    "S_front_middle-rh",
]

# ============================================================
# 3. LOAD SOURCE FILES
# ============================================================
data_dir = Path(
    r"F:\\research project\\# study 1\\eeg_data\\set"
)

# UPDATE: Path to the parent directory containing fsaverage_bem
subjects_dir = Path(
    r"F:\\research project\\# study 1\\mri_data\\fs_output\\freesurfer\\sub_dir\\Y"
)

data_dir
stc_dir = (
    data_dir / "Y" / sub_name
    / "preproc" / "analysis" / "source"
    / "morphed_stcs" / task / task_stage
)

stc_files = sorted(stc_dir.glob("*_morphed-lh.stc"))

if not stc_files:
    raise FileNotFoundError(f"No STC files found: {stc_dir}")

# Load one source estimate to identify vertices
stc = mne.read_source_estimate(str(stc_files[0]))

print("Number of epochs:", len(stc_files))
print("Total vertices:", stc.data.shape[0])

# ============================================================
# 4. MAP ROI LABELS TO SOURCE VERTICES
# ============================================================

labels = mne.read_labels_from_annot(
    subject="fsaverage_bem",
    parc="aparc.a2009s",
    subjects_dir=str(subjects_dir)
)

label_lookup = {lab.name: lab for lab in labels}

missing = set(roi_label_names) - set(label_lookup)

if missing:
    raise ValueError(f"Missing ROI labels: {sorted(missing)}")

roi_vertex_indices = {}

n_lh = len(stc.vertices[0])

for roi_name in roi_label_names:

    label = label_lookup[roi_name]

    if label.hemi == "lh":

        indices = np.where(
            np.isin(stc.vertices[0], label.vertices)
        )[0]

    else:

        indices = np.where(
            np.isin(stc.vertices[1], label.vertices)
        )[0] + n_lh

    if len(indices) == 0:
        raise ValueError(f"No vertices found for {roi_name}")

    roi_vertex_indices[roi_name] = indices

# ============================================================
# 5. CHECK ROI VERTICES
# ============================================================

roi_summary = pd.DataFrame({
    "roi": list(roi_vertex_indices.keys()),
    "n_vertices": [
        len(indices)
        for indices in roi_vertex_indices.values()
    ]
})

print("\nROI SUMMARY")
print(roi_summary.to_string(index=False))

print("\nNumber of ROIs:", len(roi_vertex_indices))

unique_vertices = np.unique(
    np.concatenate(list(roi_vertex_indices.values()))
)

print("Total unique ROI vertices:", len(unique_vertices))
print("Total source vertices:", stc.data.shape[0])


Number of epochs: 84
Total vertices: 5124
Reading labels from parcellation...
   read 75 labels from F:\research project\# study 1\mri_data\fs_output\freesurfer\sub_dir\Y\fsaverage_bem\label\lh.aparc.a2009s.annot
   read 75 labels from F:\research project\# study 1\mri_data\fs_output\freesurfer\sub_dir\Y\fsaverage_bem\label\rh.aparc.a2009s.annot

ROI SUMMARY
                     roi  n_vertices
         G_precentral-lh          60
         G_precentral-rh          59
            S_central-lh          79
            S_central-rh          75
S_precentral-sup-part-lh          28
S_precentral-sup-part-rh          33
S_precentral-inf-part-lh          31
S_precentral-inf-part-rh          29
       G_front_middle-lh          65
       G_front_middle-rh          65
  G_and_S_paracentral-lh          35
  G_and_S_paracentral-rh          37
          G_front_sup-lh         134
          G_front_sup-rh         124
        G_postcentral-lh          55
        G_postcentral-rh          48
        S_

In [4]:

from pactools import Comodulogram
import time

# ============================================================
# 1. PILOT SETTINGS
# ============================================================

pilot_rois = [
    "G_precentral-lh",
    "G_precentral-rh"
]

n_surrogates = 20

theta_range = np.linspace(4, 8, 20)
gamma_range = np.linspace(30, 80, 20)

sfreq = stc.sfreq

# Unique vertices across selected ROIs
pilot_vertices = np.unique(
    np.concatenate([
        roi_vertex_indices[roi]
        for roi in pilot_rois
    ])
)

print("Vertices to process:", len(pilot_vertices))

# ============================================================
# 2. LOAD SOURCE TIME SERIES
# ============================================================

start_time = time.time()

epoch_data = []

for file in stc_files:

    stc_epoch = mne.read_source_estimate(str(file))
    stc_epoch = stc_epoch.copy().crop(0.0, 0.495)

    epoch_data.append(
        stc_epoch.data[pilot_vertices, :]
    )

epoch_data = np.stack(epoch_data, axis=0)

# Shape: epochs x selected vertices x time
print("Data shape:", epoch_data.shape)

# ============================================================
# 3. GENERATE SURROGATE PAC
# ============================================================

n_epochs, n_vertices, n_times = epoch_data.shape

surrogate_pac = np.empty(
    (n_vertices, n_surrogates)
)

original_pac = np.empty(n_vertices)

for i in range(n_vertices):

    print(f"Vertex {i + 1}/{n_vertices}")

    data_flat = epoch_data[:, i, :].reshape(1, -1)

    estimator = Comodulogram(
        fs=sfreq,
        low_fq_range=theta_range,
        high_fq_range=gamma_range,
        method="tort",
        n_surrogates=n_surrogates,
        minimum_shift=1.0,
        random_state=42,
        progress_bar=False
    )

    estimator.fit(data_flat, data_flat)

    original_pac[i] = estimator.comod_.mean()

    surrogate_pac[i] = estimator.surrogates_.mean(
        axis=(1, 2)
    )

# ============================================================
# 4. AVERAGE WITHIN EACH ROI
# ============================================================

rows = []

for roi_name in pilot_rois:

    # Identify positions within our selected vertex array
    positions = np.where(
        np.isin(
            pilot_vertices,
            roi_vertex_indices[roi_name]
        )
    )[0]

    # ROI-average for each surrogate iteration
    roi_surrogates = surrogate_pac[positions].mean(axis=0)

    for surrogate_id, pac_value in enumerate(
        roi_surrogates, start=1
    ):

        rows.append({
            "sub": sub_name,
            "task": task,
            "task_stage": task_stage,
            "block": block,
            "condition": task + block,
            "roi": roi_name,
            "surrogate_id": surrogate_id,
            "pac_value": pac_value
        })

df_surrogate = pd.DataFrame(rows)

# ============================================================
# 5. VALIDATE AND SAVE
# ============================================================

expected_rows = len(pilot_rois) * n_surrogates

assert len(df_surrogate) == expected_rows
assert df_surrogate["pac_value"].notna().all()

output_dir = (
    data_dir / "Y group"
    / "roi_source_analysis"
    / "surrogate_analysis"
    / "pilot"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / (
    f"SURROGATE_PAC_{sub_name}{task}{task_stage}_pilot.csv"
)

if output_file.exists():
    raise FileExistsError(
        f"Pilot file already exists: {output_file}"
    )

df_surrogate.to_csv(output_file, index=False)

print("\nSURROGATE DATAFRAME")
print(df_surrogate.head())

print("\nShape:", df_surrogate.shape)
print("ROIs:", df_surrogate["roi"].nunique())
print("Surrogates:", df_surrogate["surrogate_id"].nunique())

print("\nProcessing time:",
      round((time.time() - start_time) / 60, 2),
      "minutes")

print("Saved:", output_file)


Vertices to process: 119
Data shape: (84, 119, 249)
Vertex 1/119
Vertex 2/119
Vertex 3/119
Vertex 4/119
Vertex 5/119
Vertex 6/119
Vertex 7/119
Vertex 8/119
Vertex 9/119
Vertex 10/119
Vertex 11/119
Vertex 12/119
Vertex 13/119
Vertex 14/119
Vertex 15/119
Vertex 16/119
Vertex 17/119
Vertex 18/119
Vertex 19/119
Vertex 20/119
Vertex 21/119
Vertex 22/119
Vertex 23/119
Vertex 24/119
Vertex 25/119
Vertex 26/119
Vertex 27/119
Vertex 28/119
Vertex 29/119
Vertex 30/119
Vertex 31/119
Vertex 32/119
Vertex 33/119
Vertex 34/119
Vertex 35/119
Vertex 36/119
Vertex 37/119
Vertex 38/119
Vertex 39/119
Vertex 40/119
Vertex 41/119
Vertex 42/119
Vertex 43/119
Vertex 44/119
Vertex 45/119
Vertex 46/119
Vertex 47/119
Vertex 48/119
Vertex 49/119
Vertex 50/119
Vertex 51/119
Vertex 52/119
Vertex 53/119
Vertex 54/119
Vertex 55/119
Vertex 56/119
Vertex 57/119
Vertex 58/119
Vertex 59/119
Vertex 60/119
Vertex 61/119
Vertex 62/119
Vertex 63/119
Vertex 64/119
Vertex 65/119
Vertex 66/119
Vertex 67/119
Vertex 68/119
Verte

# OPTIMISATION

In [5]:

import time
import numpy as np
from pactools import Comodulogram
from pactools.bandpass_filter import multiple_band_pass


def fast_tort_pac(
    signal,
    fs,
    theta_range,
    gamma_range,
    n_surrogates=20,
    minimum_shift=1.0,
    random_state=42
):
    """
    Calculate original and surrogate Tort MI.

    Returns
    -------
    pac_values : ndarray
        Shape (n_surrogates + 1,).
        First value = original PAC.
        Remaining values = surrogate PAC.
    """

    signal = np.asarray(signal).reshape(1, -1)

    # --------------------------------------------------------
    # 1. Filter once using pactools
    # --------------------------------------------------------

    filtered_low = multiple_band_pass(
        signal, fs, theta_range, 2.0
    )

    filtered_high = multiple_band_pass(
        signal, fs, gamma_range, 2 * max(theta_range)
    )

    # Theta phase
    phases = np.angle(
        filtered_low.reshape(len(theta_range), -1)
    )

    # Gamma amplitude
    amplitudes = np.abs(
        filtered_high.reshape(len(gamma_range), -1)
    )

    n_points = signal.shape[1]
    n_bins = 18

    # --------------------------------------------------------
    # 2. Reproduce pactools' random shifts
    # --------------------------------------------------------

    rng = np.random.RandomState(random_state)

    min_samples = max(1, int(fs * minimum_shift))

    if n_points < 2 * min_samples:
        raise ValueError("Signal too short for minimum_shift")

    shifts = rng.randint(
        min_samples,
        n_points - min_samples,
        size=n_surrogates + 1
    )

    shifts[0] = 0

    # --------------------------------------------------------
    # 3. Calculate Tort MI
    # --------------------------------------------------------

    pac_values = np.zeros(n_surrogates + 1)

    eps = np.finfo(phases.dtype).eps * 2
    phase_edges = np.linspace(
        -np.pi, np.pi + eps, n_bins + 1
    )

    for phase in phases:

        # Assign theta phase to 18 bins
        phase_bins = np.digitize(
            phase, phase_edges
        ) - 1

        for s, shift in enumerate(shifts):

            shifted_bins = np.roll(phase_bins, shift)

            counts = np.bincount(
                shifted_bins,
                minlength=n_bins
            )

            for amplitude in amplitudes:

                amp_sum = np.bincount(
                    shifted_bins,
                    weights=amplitude,
                    minlength=n_bins
                )

                # Same empty-bin convention as pactools
                amp_dist = np.ones(n_bins)

                valid = counts > 0

                amp_dist[valid] = (
                    amp_sum[valid] / counts[valid]
                )

                amp_dist /= amp_dist.sum()

                kl = np.sum(
                    amp_dist * np.log(amp_dist * n_bins)
                )

                mi = kl / np.log(n_bins)

                pac_values[s] += mi

    pac_values /= (
        len(theta_range) * len(gamma_range)
    )

    return pac_values


In [6]:

# Select the first vertex
signal = epoch_data[:, 0, :].reshape(1, -1)

# ============================================================
# ORIGINAL PACTOOLS
# ============================================================

start = time.perf_counter()

estimator = Comodulogram(
    fs=sfreq,
    low_fq_range=theta_range,
    high_fq_range=gamma_range,
    method="tort",
    n_surrogates=20,
    minimum_shift=1.0,
    random_state=42,
    progress_bar=False
)

estimator.fit(signal, signal)

pactools_values = np.concatenate([
    [estimator.comod_.mean()],
    estimator.surrogates_.mean(axis=(1, 2))
])

time_pactools = time.perf_counter() - start

# ============================================================
# OPTIMIZED IMPLEMENTATION
# ============================================================

start = time.perf_counter()

fast_values = fast_tort_pac(
    signal,
    fs=sfreq,
    theta_range=theta_range,
    gamma_range=gamma_range,
    n_surrogates=20,
    minimum_shift=1.0,
    random_state=42
)

time_fast = time.perf_counter() - start

# ============================================================
# COMPARE
# ============================================================

print("Original PAC (pactools):", pactools_values[0])
print("Original PAC (optimized):", fast_values[0])

print("\nMaximum absolute difference:")
print(np.max(np.abs(pactools_values - fast_values)))

print("\nAll values match:")
print(np.allclose(
    pactools_values,
    fast_values,
    rtol=1e-6,
    atol=1e-12
))

print("\nProcessing times:")
print(f"Pactools: {time_pactools:.2f} seconds")
print(f"Optimized: {time_fast:.2f} seconds")

print(
    f"Speedup: {time_pactools / time_fast:.2f}x"
)


Original PAC (pactools): 0.00015457353320203388
Original PAC (optimized): 0.00015457352781375025

Maximum absolute difference:
2.7077076041395753e-11

All values match:
True

Processing times:
Pactools: 3.28 seconds
Optimized: 0.33 seconds
Speedup: 9.79x


In [7]:
# Calculate optimized PAC for all pilot vertices
fast_results = np.array([
    fast_tort_pac(
        epoch_data[:, i, :],
        fs=sfreq,
        theta_range=theta_range,
        gamma_range=gamma_range,
        n_surrogates=20,
        minimum_shift=1.0,
        random_state=42
    )
    for i in range(len(pilot_vertices))
])

print("Fast results shape:", fast_results.shape)

# Compare original and surrogate PAC
print(
    "Maximum vertex-level difference:",
    np.max(np.abs(fast_results[:, 0] - original_pac))
)

print(
    "Maximum surrogate difference:",
    np.max(np.abs(fast_results[:, 1:] - surrogate_pac))
)

Fast results shape: (119, 21)
Maximum vertex-level difference: 1.07362220361023e-08
Maximum surrogate difference: 2.8583570888315998e-08


In [8]:
# Relative errors, with protection against division by zero
original_rel_error = np.abs(
    fast_results[:, 0] - original_pac
) / np.maximum(np.abs(original_pac), 1e-12)

surrogate_rel_error = np.abs(
    fast_results[:, 1:] - surrogate_pac
) / np.maximum(np.abs(surrogate_pac), 1e-12)

print(
    "Maximum original relative error (%):",
    original_rel_error.max() * 100
)

print(
    "Maximum surrogate relative error (%):",
    surrogate_rel_error.max() * 100
)

print(
    "Mean surrogate relative error (%):",
    surrogate_rel_error.mean() * 100
)

Maximum original relative error (%): 0.007052049243616863
Maximum surrogate relative error (%): 0.021791864012336407
Mean surrogate relative error (%): 0.0002838921384599938


**What we've accomplished**
- Reproduced the original PAC estimates.
- Generated time-shifted surrogates.
- Completed preliminary simulation checks.
- Verified all 24 ROI labels (1,309 unique vertices).
- Created a surrogate dataframe compatible with your R model.
- Achieved approximately 9.8× faster computation.
- Confirmed close numerical agreement across 119 vertices.

**VALIDATION for all 24 ROIs of one participant in one condition**

Step 1. Load the data for all ROI vertices

In [9]:

import numpy as np
import pandas as pd
import time
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

sub_name = "s1_pac_sub01"
task = "_BL"
task_stage = "_go"
block = ""

n_surrogates = 20

# All 24 ROIs
selected_rois = roi_label_names

# Unique vertices across all selected ROIs
all_roi_vertices = np.unique(
    np.concatenate([
        roi_vertex_indices[roi]
        for roi in selected_rois
    ])
)

print("Number of ROIs:", len(selected_rois))
print("Unique vertices:", len(all_roi_vertices))

# ============================================================
# LOAD SOURCE DATA
# ============================================================

start_time = time.time()

epoch_data_all = []

for file in stc_files:

    stc_epoch = mne.read_source_estimate(str(file))
    stc_epoch = stc_epoch.copy().crop(0.0, 0.495)

    epoch_data_all.append(
        stc_epoch.data[all_roi_vertices, :]
    )

epoch_data_all = np.stack(epoch_data_all, axis=0)

print("Data shape:", epoch_data_all.shape)
print(
    "Loading time:",
    round((time.time() - start_time) / 60, 2),
    "minutes"
)


Number of ROIs: 24
Unique vertices: 1309
Data shape: (84, 1309, 249)
Loading time: 0.02 minutes


Step 2. Calculate surrogate PAC with checkpoints

In [10]:

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

output_dir = (
    data_dir / "Y group"
    / "roi_source_analysis"
    / "surrogate_analysis"
    / "full_roi_pilot"
)

output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_file = output_dir / (
    f"CHECKPOINT_{sub_name}{task}{task_stage}.npz"
)

# ============================================================
# INITIALIZE OR RESUME
# ============================================================

n_vertices = len(all_roi_vertices)

all_pac_results = np.full(
    (n_vertices, n_surrogates + 1),
    np.nan
)

start_vertex = 0

if checkpoint_file.exists():

    checkpoint = np.load(checkpoint_file)

    saved_vertices = checkpoint["vertex_indices"]
    saved_results = checkpoint["pac_results"]

    if not np.array_equal(saved_vertices, all_roi_vertices):
        raise ValueError("Checkpoint vertex indices do not match")

    if saved_results.shape != all_pac_results.shape:
        raise ValueError("Checkpoint shape does not match settings")

    all_pac_results = saved_results.copy()

    completed = np.all(
        np.isfinite(all_pac_results),
        axis=1
    )

    start_vertex = np.where(~completed)[0]

    start_vertex = (
        start_vertex[0]
        if len(start_vertex) > 0
        else n_vertices
    )

    print(f"Resuming from vertex {start_vertex + 1}")

# ============================================================
# CALCULATE PAC
# ============================================================

start_time = time.time()

for i in range(start_vertex, n_vertices):

    signal = epoch_data_all[:, i, :]

    all_pac_results[i] = fast_tort_pac(
        signal,
        fs=sfreq,
        theta_range=theta_range,
        gamma_range=gamma_range,
        n_surrogates=n_surrogates,
        minimum_shift=1.0,
        random_state=42
    )

    # Save checkpoint every 100 vertices
    if (i + 1) % 100 == 0:

        np.savez(
            checkpoint_file,
            vertex_indices=all_roi_vertices,
            pac_results=all_pac_results
        )

        elapsed = (time.time() - start_time) / 60

        print(
            f"Completed {i + 1}/{n_vertices} vertices "
            f"({elapsed:.1f} min)"
        )

# Final checkpoint
np.savez(
    checkpoint_file,
    vertex_indices=all_roi_vertices,
    pac_results=all_pac_results
)

print("\nPAC computation complete!")

print(
    "Processing time:",
    round((time.time() - start_time) / 60, 2),
    "minutes"
)

print("PAC results shape:", all_pac_results.shape)


Completed 100/1309 vertices (0.9 min)
Completed 200/1309 vertices (1.7 min)
Completed 300/1309 vertices (2.6 min)
Completed 400/1309 vertices (3.2 min)
Completed 500/1309 vertices (3.7 min)
Completed 600/1309 vertices (4.2 min)
Completed 700/1309 vertices (4.8 min)
Completed 800/1309 vertices (5.3 min)
Completed 900/1309 vertices (5.9 min)


C:\Users\a1902989\AppData\Local\Temp\ipykernel_14880\425402850.py:120: RuntimeWarning: divide by zero encountered in log
  amp_dist * np.log(amp_dist * n_bins)
C:\Users\a1902989\AppData\Local\Temp\ipykernel_14880\425402850.py:120: RuntimeWarning: invalid value encountered in multiply
  amp_dist * np.log(amp_dist * n_bins)


Completed 1000/1309 vertices (6.4 min)
Completed 1100/1309 vertices (7.0 min)
Completed 1200/1309 vertices (7.5 min)
Completed 1300/1309 vertices (8.1 min)

PAC computation complete!
Processing time: 8.12 minutes
PAC results shape: (1309, 21)


Step 3. Average across vertices and save ROI results

In [13]:

# ============================================================
# CONSTRUCT ROI-LEVEL DATAFRAME
# ============================================================

rows = []

for roi_name in selected_rois:

    # Find positions of this ROI's vertices
    positions = np.where(
        np.isin(
            all_roi_vertices,
            roi_vertex_indices[roi_name]
        )
    )[0]

    # Average over vertices
    roi_values = all_pac_results[positions].mean(axis=0)

    # Observed PAC
    original_value = roi_values[0]

    # 20 surrogate values
    for surrogate_id in range(1, n_surrogates + 1):

        rows.append({
            "sub": sub_name,
            "task": task,
            "task_stage": task_stage,
            "block": block,
            "condition": task + block,
            "roi": roi_name,
            "surrogate_id": surrogate_id,
            "pac_value": roi_values[surrogate_id],
            "original_pac": original_value
        })

df_surrogate_all = pd.DataFrame(rows)

# ============================================================
# ROI AVERAGING - EXCLUDE INVALID VERTICES
# ============================================================

rows = []

for roi_name in selected_rois:

    positions = np.where(
        np.isin(
            all_roi_vertices,
            roi_vertex_indices[roi_name]
        )
    )[0]

    # Keep only vertices with valid original AND surrogate PAC
    valid_positions = positions[
        np.isfinite(all_pac_results[positions]).all(axis=1)
    ]

    if len(valid_positions) == 0:
        raise ValueError(f"No valid vertices in {roi_name}")

    roi_values = all_pac_results[valid_positions].mean(axis=0)

    original_value = roi_values[0]

    for surrogate_id in range(1, n_surrogates + 1):

        rows.append({
            "sub": sub_name,
            "task": task,
            "task_stage": task_stage,
            "block": block,
            "condition": task + block,
            "roi": roi_name,
            "surrogate_id": surrogate_id,
            "pac_value": roi_values[surrogate_id],
            "original_pac": original_value
        })

df_surrogate_all = pd.DataFrame(rows)

# ============================================================
# VALIDATE
# ============================================================

assert len(df_surrogate_all) == 24 * n_surrogates
assert df_surrogate_all["pac_value"].notna().all()
assert df_surrogate_all["roi"].nunique() == 24

print("Dataframe shape:", df_surrogate_all.shape)
print("ROIs:", df_surrogate_all["roi"].nunique())
print("Surrogates:", df_surrogate_all["surrogate_id"].nunique())


# ============================================================
# SAVE
# ============================================================

output_file = output_dir / (
    f"SURROGATE_PAC_{sub_name}{task}{task_stage}_allROIs.csv"
)

if output_file.exists():
    raise FileExistsError(
        f"Output already exists: {output_file}"
    )

df_surrogate_all.to_csv(output_file, index=False)

print("\nSaved:", output_file)
print(df_surrogate_all.head())


Dataframe shape: (480, 9)
ROIs: 24
Surrogates: 20

Saved: F:\research project\# study 1\eeg_data\set\Y group\roi_source_analysis\surrogate_analysis\full_roi_pilot\SURROGATE_PAC_s1_pac_sub01_BL_go_allROIs.csv
            sub task task_stage block condition              roi  \
0  s1_pac_sub01  _BL        _go             _BL  G_precentral-lh   
1  s1_pac_sub01  _BL        _go             _BL  G_precentral-lh   
2  s1_pac_sub01  _BL        _go             _BL  G_precentral-lh   
3  s1_pac_sub01  _BL        _go             _BL  G_precentral-lh   
4  s1_pac_sub01  _BL        _go             _BL  G_precentral-lh   

   surrogate_id  pac_value  original_pac  
0             1   0.000157      0.000163  
1             2   0.000168      0.000163  
2             3   0.000174      0.000163  
3             4   0.000157      0.000163  
4             5   0.000162      0.000163  


In [14]:
# ============================================================
# COMPARE ROI PAC WITH ORIGINAL SAVED RESULTS
# ============================================================

original_roi_file = (
    data_dir / "Y group"
    / "roi_source_analysis"
    / "PAC_MI_SOURCE_Y_ROI.csv"
)

df_original = pd.read_csv(original_roi_file)

affected_rois = [
    "G_front_middle-rh",
    "S_front_middle-rh"
]

for roi_name in affected_rois:

    # Original ROI PAC (from previously saved analysis)
    original_match = df_original[
        (df_original["sub"] == sub_name)
        & (df_original["task"] == task)
        & (df_original["task_stage"] == task_stage)
        & (df_original["roi"] == roi_name)
    ]

    if len(original_match) != 1:
        raise ValueError(
            f"Expected one original row for {roi_name}, "
            f"found {len(original_match)}"
        )

    original_value = original_match["pac_value"].iloc[0]

    # Recalculated PAC after excluding invalid vertices
    new_value = df_surrogate_all.loc[
        df_surrogate_all["roi"] == roi_name,
        "original_pac"
    ].iloc[0]

    relative_difference = (
        abs(new_value - original_value)
        / abs(original_value)
        * 100
    )

    print(f"\nROI: {roi_name}")
    print("Original PAC (imputed):", original_value)
    print("New PAC (excluded):", new_value)
    print("Relative difference (%):", relative_difference)


ROI: G_front_middle-rh
Original PAC (imputed): 0.0001503022984783
New PAC (excluded): 0.0001475337252789382
Relative difference (%): 1.8420032344092823

ROI: S_front_middle-rh
Original PAC (imputed): 0.0001630182126422
New PAC (excluded): 0.00013108253378906745
Relative difference (%): 19.59025211693768


In [15]:
# Full source-space indices of invalid vertices
invalid_indices = all_roi_vertices[
    ~np.isfinite(all_pac_results).all(axis=1)
]

print("Invalid vertex indices:", invalid_indices)


Invalid vertex indices: [3457 4029]
